# A3.1 · Default-deny on the tool call

**Function A — Securing AI Architectures → Securing the Architecture — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A2.8 · An audit trail the workload cannot forge](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**.

| | |
|---|---|
| Tools used | OPA / Rego, SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Evaluate the same call under allow-by-default and deny-by-default policy and compare what gets through.

**Why a security engineer needs it.** Allow-by-default authorization is defeated by any argument the model can be persuaded to produce. The control it builds is: policy evaluated per call on (identity, tool, arguments, resource), denying unless a rule permits.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Identity has already failed. Something untrusted is in the context and the agent has decided to call a tool. The tool call is the last place a decision can still be made on facts rather than intent — this SPIFFE ID, this tool, this resource, this verb — and a policy written one notch vaguer than that cannot express the distinction the attack turns on.

> **At CyberTravels.** The last place a decision about that refund rests on facts rather than on intent. Identity has already failed, an injected instruction is in the context, and the tool call is where CyberTravels can still say no. R1, R3.

## 2 · The framework

```
   untrusted text in context ---> agent decides to call a tool
                                             |
                                    +--------v---------+
                                    |  policy decision |
                                    |  DEFAULT: DENY   |
                                    +--------+---------+
                                             |
                        allow only on facts: identity, scope,
                        resource, provenance of the motivating span

   the last point where a decision rests on facts rather than on intent
```

**Mitigates: T2 Tool Misuse · T3 Privilege Compromise · T6 Intent Breaking.**

The tool call is the moment text becomes consequence. It is also the last point
where a decision can be made on **facts** — this identity, this tool, these
arguments, this resource — rather than on intent, which nobody can read.

**Start from the identity, and be very specific about it.** Not "the Workflow
Agent"; the SPIFFE ID A2.3 issued —
`spiffe://cybertravels.com/ns/prod/sa/workflow-agent` — and against it, an
entitlement written at full resolution: which tools, on which resources, with
which verbs. Everything else in this lesson is a consequence of writing the
entitlement down at that resolution. A policy phrased one notch vaguer cannot
express the distinction the attack turns on, and the vagueness is invisible
until it is exploited.

Default-deny then means the absence of a rule is a refusal. That sounds like a
detail and it is the entire control, because it changes what a mistake costs.
Under allow-by-default, a permission somebody forgot to restrict is available to
an attacker. Under deny-by-default, a permission somebody forgot to grant is a
broken feature — which someone reports on Monday morning, loudly, and which
harms nobody.

The policy takes four inputs and all four matter:

- **identity** — the attested workload identity, from A2.3
- **tool** — which capability
- **arguments** — the actual values, not the schema
- **resource** — which specific thing

Dropping the fourth is the most common weakening. `run_query` permitted for the
Workflow Agent is not the same as `run_query` permitted *on the bookings table*,
and A1.5 was the difference between those two sentences. The verb is the second
most common: `charge_card` entitled for payments is not `refund` entitled for
payments, and R1 is the entire distance between them.

This does not stop the agent being persuaded. It stops persuasion mattering,
which is a better place to stand.

> **What this control closes.**
>
> Stands on the edge every topology shares: `agent_runtime -> tools`. Persuasion still happens; it just stops reaching anything.

## 3 · Proving the baseline is default-deny, as a skill

Writing the entitlement is one job; showing that CyberTravels' running role *is* the entitlement is another, and it is the one an auditor asks for. The procedure reads every inline and attached policy for the baseline, then measures granted-but-unused permissions against what the audit trail observed — which only means anything if the trail is intact, so incomplete coverage is a finding rather than a clean pass. This is the file in this repository:

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/attestation/iam-least-privilege-verifier/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: iam-least-privilege-verifier
description: >-
  Prove a deployment's role baseline is default-deny and quantify granted-
  but-unused permissions against observed usage. Use to evidence least
  privilege, to find excess permissions on an agent role, or when asked
  whether a deployment's IAM posture supports a default-deny claim.
allowed-tools: Bash, Read
---

# Iam Least Privilege Verifier

**Controls:** Control 1 — default-deny and least privilege

## Confidence: HIGH

This is one of the controls that is genuinely provable at runtime. Policy
documents are readable, and usage data turns "least privilege" from an
assertion into a measured delta.

## Procedure

1. **Establish the default-deny baseline.** Read every inline and attached
   policy. The baseline fails if any of these are present:
   - `Action: "*"` or `Resource: "*"` in an Allow statement
   - broad managed policies such as administrator or power-user equivalents
   - a wildcard principal on a trust policy

2. **Measure excess.** Compare granted permissions against observed usage from
   the access-analysis and last-accessed data. Count actions, roles, keys and
   passwords idle for at least the tracking period (configurable 1–180 days;
   default 90).

3. **Generate the least-privilege diff.** Policy generation derived from actual
   activity produces a candidate policy; the diff against what is granted is
   the excess-permission finding, expressed concretely rather than as a score.

4. **Check external access.** External-access findings must be zero, or each
   one must map to an approved exception.

## Output contract

```json
{
  "deployment_id": "str",
  "role_arn": "str",
  "default_deny_verified": true,
  "wildcard_findings": [{"policy": "str", "statement": "str"}],
  "excess_permission_count": 0,
  "unused": [{"type": "action|role|key|password", "name": "str", "idle_days": 0}],
  "generated_policy_diff": "str",
  "external_access_findings": 0,
  "tracking_period_days": 90,
  "verdict": "PASS|PARTIAL|FAIL"
}
```

## Failure modes

- **Incomplete audit-trail coverage.** If the trail is not intact, "unused" is
  unreliable and the verdict must be `PARTIAL`. Policy generation can miss
  legitimately-used-but-rare actions — an annual disaster-recovery permission
  looks identical to dead permission over a 90-day window.
- **Counting managed-policy names instead of effective actions.** Two policies
  can grant the same action; the union is what matters.
- **Treating a low excess count as a pass** while a wildcard is present. The
  baseline check is a gate, not a contributor to a score.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## What you just proved

The skill loads and reports its shape. Two of its failure modes are the ones this lesson is about: counting managed-policy *names* instead of effective actions, and reading a low excess count as a pass while a wildcard sits in the policy — a wildcard is not a large number of permissions, it is an unbounded one.

## Your turn

Take one tool policy you have and check whether it names the resource *and* the verb. If it grants `run_query` rather than `SELECT on these tables`, it cannot express the difference that A1.5 and R1 both turn on.

---

**Next → [A3.2 · Sandboxed execution](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*